In [1]:
import pandas as pd
import numpy as np
from docx import Document
from docx.oxml.ns import qn
from collections import Counter

In [2]:
DOCX_PATH = 'SA/Industry-premium-rates-2025-26.docx'

SECTION_NAMES = {
    "Agriculture, forestry and fishing", "Mining", "Manufacturing",
    "Electricity, Gas, Water and Waste Services", "Construction",
    "Wholesale trade", "Retail trade", "Accommodation and food services",
    "Transport, postal and warehousing", "Information, media and telecommunications",
    "Financial and insurance services", "Rental, hiring and real estate services",
    "Professional, scientific and technical services", "Administrative and support services",
    "Public administration and safety", "Education and training",
    "Health care and social assistance", "Arts and recreation services",
    "Other services", "Non-classifiable"
}

doc = Document(DOCX_PATH)
body = doc.element.body

# Map each table index -> its preceding section heading
section_map = {}
current_section = "Unknown"
table_idx = 0

for elem in body:
    tag = elem.tag.split('}')[-1]
    if tag == 'p':
        para_text = ''.join(r.text for r in elem.iter(qn('w:t'))).strip()
        if para_text in SECTION_NAMES:
            current_section = para_text
    elif tag == 'tbl':
        section_map[table_idx] = current_section
        table_idx += 1

# Extract all rows from all tables
rows = []
for i, table in enumerate(doc.tables):
    section = section_map.get(i, "Unknown")
    for row in table.rows:
        cells = [cell.text.strip() for cell in row.cells]
        if cells[0].lower() == 'saic number' or not any(cells):
            continue
        rows.append([section] + cells)

df = pd.DataFrame(rows, columns=['section', 'saic_number', 'industry_description', 'premium_rate_per_100'])

# Clean up the rate column - strip the % sign and convert to float
df['premium_rate'] = df['premium_rate_per_100'].str.replace('%', '', regex=False).str.strip().astype(float)

print(f"{len(df)} rows across {df['section'].nunique()} sections")
df.head()

528 rows across 20 sections


,section,saic_number,industry_description,premium_rate_per_100,premium_rate
0,"Agriculture, forestry and fishing",011101,Nursery Production,3.247%,3.247
1,"Agriculture, forestry and fishing",011301,Turf Growing,3.452%,3.452
2,"Agriculture, forestry and fishing",011401,Floriculture Production,3.703%,3.703
3,"Agriculture, forestry and fishing",012101,Mushroom Growing,3.761%,3.761
4,"Agriculture, forestry and fishing",012201,Vegetable Growing,3.758%,3.758


In [3]:
df.groupby('section')['saic_number'].count().sort_values(ascending=False)

section
Manufacturing                                      148
Wholesale trade                                     47
Retail trade                                        42
Agriculture, forestry and fishing                   34
Transport, postal and warehousing                   29
Other services                                      27
Information, media and telecommunications           24
Construction                                        23
Health care and social assistance                   20
Arts and recreation services                        19
Mining                                              17
Professional, scientific and technical services     16
Financial and insurance services                    14
Administrative and support services                 14
Public administration and safety                    14
Education and training                              11
Rental, hiring and real estate services             11
Electricity, Gas, Water and Waste Services          11
Ac

In [4]:
# Export to CSV and Parquet
df.to_csv('SA/industry_premium_rates_2025-26.csv', index=False)
df.to_parquet('SA/industry_premium_rates_2025-26.parquet', index=False)
print("Saved SA/industry_premium_rates_2025-26.csv")
print("Saved SA/industry_premium_rates_2025-26.parquet")

Saved SA/industry_premium_rates_2025-26.csv
Saved SA/industry_premium_rates_2025-26.parquet


In [5]:
df.describe()

,premium_rate
count,528.000000
mean,2.642568
std,2.195791
min,0.400000
25%,1.104750
50%,2.356000
75%,3.595500
max,23.812000
